In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

### Load all finalized datasets for final merge

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Final aggregated fire dataset
fire_df = pd.read_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Merge_Fire_Dataset.csv')
fire = fire_df.copy()

# Final core meteorological dataset
core_df = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Weather_Core_LC_Dataset.csv")
core = core_df.copy()

# Final gust meteorological dataset
gust_df = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Weather_Gust_LC_Dataset.csv")
gust = gust_df.copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-1021935457.py:9: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  core_df = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Weather_Core_LC_Dataset.csv")
/tmp/ipython-input-1021935457.py:13: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  gust_df = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Weather_Gust_LC_Dataset.csv")


### Prepare merge columns

In [ ]:
print("Fire Columns:", fire.columns)
print("___________________________________________________________________________________")
print("Core Columns:", core.columns)
print("___________________________________________________________________________________")
print("Gust Columns:", gust.columns)

Fire Columns: Index(['Fire_ID', 'Acquisition_Date', 'Acquisition_Time', 'Year', 'Month',
       'Day_of_Year', 'Hour', 'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4',
       'Bright_TI5', 'FRP', 'Scan', 'Track', 'Confidence', 'DayNight',
       'Fire_Type', 'LC_Type1', 'LC_Label', 'Confidence_Ordered',
       'detection_count', 'FRP_max'],
      dtype='object')
___________________________________________________________________________________
Core Columns: Index(['Station_Name', 'Province', 'Climate_ID', 'Elevation_m',
       'Latitude_Weather', 'Longitude_Weather', 'Date', 'LC_Type1', 'LC_Label',
       'Max_Temp_C', 'Min_Temp_C', 'Mean_Temp_C', 'Total_Precip_mm'],
      dtype='object')
___________________________________________________________________________________
Gust Columns: Index(['Station_Name', 'Province', 'Climate_ID', 'Elevation_m',
       'Latitude_Weather', 'Longitude_Weather', 'Date', 'LC_Type1', 'LC_Label',
       'Max_Temp_C', 'Min_Temp_C', 'Mean_Temp_C', 'Total_Pr

In [ ]:
# Prepare fire date column
fire["Date"] = pd.to_datetime(fire["Acquisition_Date"])

# Prepare weather date columns
core["Date"] = pd.to_datetime(core["Date"])
gust["Date"] = pd.to_datetime(gust["Date"])

### Build CORE weather station lookup table and find nearest station for each fire point

In [ ]:
# Unique stations from core (one row per Climate_ID)
core_stations = (
    core[["Climate_ID", "Station_Name", "Province", "Elevation_m",
          "Latitude_Weather", "Longitude_Weather"]]
    .drop_duplicates(subset=["Climate_ID"])
    .reset_index(drop=True)
)

# Convert coordinates to radians
core_station_coords_rad = np.radians(core_stations[["Latitude_Weather", "Longitude_Weather"]].values)
core_fire_coords_rad = np.radians(fire[["Latitude_Fire", "Longitude_Fire"]].values)

tree = BallTree(core_station_coords_rad, metric="haversine")

# Query nearest station (k = 1)
dist_rad, idx = tree.query(core_fire_coords_rad, k=1)

# Convert radians to km
earth_radius_km = 6371.0088
fire["Nearest_Core_Station_Dist_km"] = dist_rad[:, 0] * earth_radius_km
fire["Nearest_Core_Station_Index"] = idx[:, 0]

# Combine station IDs and metadata
fire = fire.join(
    core_stations.loc[fire["Nearest_Core_Station_Index"],
     ["Climate_ID", "Station_Name", "Province", "Elevation_m", "Latitude_Weather", "Longitude_Weather"]]
    .reset_index(drop=True)
)


In [ ]:
fire.head()

,Fire_ID,Acquisition_Date,Acquisition_Time,Year,Month,Day_of_Year,Hour,Latitude_Fire,Longitude_Fire,Bright_TI4,...,FRP_max,Date,Nearest_Core_Station_Dist_km,Nearest_Core_Station_Index,Climate_ID,Station_Name,Province,Elevation_m,Latitude_Weather,Longitude_Weather
0,2255440,2024-08-11,1114,2024,8,224,11,48.30000,-120.57123,308.78,...,2.70,2024-08-11,116.147517,199,1125852,OSOYOOS CS,BRITISH COLUMBIA,282.9,49.03,-119.44
1,935127,2019-10-09,2152,2019,10,282,21,48.30000,-118.81944,356.92,...,164.97,2019-10-09,77.921241,210,1135126,MIDWAY,BRITISH COLUMBIA,571.0,49.00,-118.77
2,251428,2015-08-23,1003,2015,8,235,10,48.30001,-118.86491,332.71,...,2.09,2015-08-23,78.147078,210,1135126,MIDWAY,BRITISH COLUMBIA,571.0,49.00,-118.77
3,1024549,2021-07-18,1114,2021,7,199,11,48.30001,-118.53941,317.18,...,13.97,2021-07-18,79.657312,210,1135126,MIDWAY,BRITISH COLUMBIA,571.0,49.00,-118.77
4,660545,2018-08-14,1018,2018,8,226,10,48.30001,-116.10197,296.26,...,1.35,2018-08-14,85.323452,218,1144635,LISTER,BRITISH COLUMBIA,660.0,49.03,-116.46


### Build GUST weather station lookup table and find nearest station for each fire point

In [ ]:
# Start with new copy of fire df
fire_gust = fire_df.copy()
fire_gust["Date"] = pd.to_datetime(fire_gust["Acquisition_Date"])

# Unique stations from gust (one row per Climate_ID)
gust_stations = (
    gust[["Climate_ID", "Station_Name", "Province", "Elevation_m",
          "Latitude_Weather", "Longitude_Weather"]]
    .drop_duplicates(subset=["Climate_ID"])
    .reset_index(drop=True)
)

# Convert coordinates to radians
gust_station_coords_rad = np.radians(gust_stations[["Latitude_Weather", "Longitude_Weather"]].values)
fire_coords_rad = np.radians(fire_gust[["Latitude_Fire", "Longitude_Fire"]].values)

tree = BallTree(gust_station_coords_rad, metric="haversine")

# Query nearest station (k = 1)
dist_rad, idx = tree.query(fire_coords_rad, k=1)

# Convert radians to km
earth_radius_km = 6371.0088
fire_gust["Nearest_Gust_Station_Dist_km"] = dist_rad[:, 0] * earth_radius_km
fire_gust["Nearest_Gust_Station_Index"] = idx[:, 0]

# Combine station IDs and metadata
fire_gust = fire_gust.join(
    gust_stations.loc[fire_gust["Nearest_Gust_Station_Index"],
     ["Climate_ID", "Station_Name", "Province", "Elevation_m", "Latitude_Weather", "Longitude_Weather"]]
    .reset_index(drop=True)
)


In [ ]:
fire_gust.head()

,Fire_ID,Acquisition_Date,Acquisition_Time,Year,Month,Day_of_Year,Hour,Latitude_Fire,Longitude_Fire,Bright_TI4,...,FRP_max,Date,Nearest_Gust_Station_Dist_km,Nearest_Gust_Station_Index,Climate_ID,Station_Name,Province,Elevation_m,Latitude_Weather,Longitude_Weather
0,2255440,2024-08-11,1114,2024,8,224,11,48.30000,-120.57123,308.78,...,2.70,2024-08-11,116.147517,139,1125852,OSOYOOS CS,BRITISH COLUMBIA,282.9,49.03,-119.44
1,935127,2019-10-09,2152,2019,10,282,21,48.30000,-118.81944,356.92,...,164.97,2019-10-09,93.090234,139,1125852,OSOYOOS CS,BRITISH COLUMBIA,282.9,49.03,-119.44
2,251428,2015-08-23,1003,2015,8,235,10,48.30001,-118.86491,332.71,...,2.09,2015-08-23,91.500890,139,1125852,OSOYOOS CS,BRITISH COLUMBIA,282.9,49.03,-119.44
3,1024549,2021-07-18,1114,2021,7,199,11,48.30001,-118.53941,317.18,...,13.97,2021-07-18,104.703604,139,1125852,OSOYOOS CS,BRITISH COLUMBIA,282.9,49.03,-119.44
4,660545,2018-08-14,1018,2018,8,226,10,48.30001,-116.10197,296.26,...,1.35,2018-08-14,85.323452,151,1144635,LISTER,BRITISH COLUMBIA,660.0,49.03,-116.46


### Merge meteorological values by (Climate_ID, Date)

In [ ]:
# Define core merge columns to avoid column duplication
core_merge = core[["Climate_ID", "Date", "Max_Temp_C", "Min_Temp_C", "Mean_Temp_C", "Total_Precip_mm",
                   "LC_Type1", "LC_Label"]].copy()
core_merge["Date"] = pd.to_datetime(core_merge["Date"])

# Merge fire and core datasets
fire_core_merged = fire.merge(
    core_merge,
    on=["Climate_ID", "Date"],
    how="left"
)


Rows: 2300610
Nearest core station distance (km):
count    2.300610e+06
mean     7.227808e+01
std      4.574423e+01
min      1.569962e-02
50%      6.269114e+01
90%      1.411777e+02
95%      1.553627e+02
99%      1.875637e+02
max      2.801708e+02
Name: Nearest_Core_Station_Dist_km, dtype: float64
0


In [ ]:
# Quick validation checks
print("Rows:", len(fire_core_merged))
print("Missing temp after merge (Mean_Temp_C):",
      fire_core_merged["Mean_Temp_C"].isna().sum())

print("Nearest core station distance (km):")
print(fire_core_merged["Nearest_Core_Station_Dist_km"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# Should be 0 if all Climate_ID values exist in core_stations
print((~fire_core_merged["Climate_ID"].isin(core_stations["Climate_ID"])).sum())

Rows: 2300610
Missing temp after merge (Mean_Temp_C): 737530
Nearest core station distance (km):
count    2.300610e+06
mean     7.227808e+01
std      4.574423e+01
min      1.569962e-02
50%      6.269114e+01
90%      1.411777e+02
95%      1.553627e+02
99%      1.875637e+02
max      2.801708e+02
Name: Nearest_Core_Station_Dist_km, dtype: float64
0


In [ ]:
fire_core_merged.groupby(
    fire_core_merged["Mean_Temp_C"].isna()
)["Nearest_Core_Station_Dist_km"].describe()

print("Missing Mean_Temp_C within 50 km:",
      fire_core_merged.loc[
          fire_core_merged["Nearest_Core_Station_Dist_km"] <= 50,
          "Mean_Temp_C"
      ].isna().mean())

print("Missing Mean_Temp_C beyond 150 km:",
      fire_core_merged.loc[
          fire_core_merged["Nearest_Core_Station_Dist_km"] >= 150,
          "Mean_Temp_C"
      ].isna().mean())

Missing Mean_Temp_C within 50 km: 0.4120514764670323
Missing Mean_Temp_C beyond 150 km: 0.14889043225187684


In [ ]:
# Define gust merge columns to avoid column duplication
gust_merge = gust[["Climate_ID", "Date", "Max_Temp_C", "Min_Temp_C", "Mean_Temp_C", "Total_Precip_mm",
                   "Max_Gust_Speed_kmh", "Gust_Flag_Threshold", "Gust_Flag_Imputed", "Imputed_Radius_km",
                   "LC_Type1", "LC_Label"]].copy()
gust_merge["Date"] = pd.to_datetime(gust_merge["Date"])

fire_gust_merged = fire_gust.merge(
    gust_merge,
    on=["Climate_ID", "Date"],
    how="left"
)

# Quick validation checks
print("Rows:", len(fire_gust_merged))
print("Missing gust after merge (Max_Gust_Speed_kmh):",
      fire_gust_merged["Max_Gust_Speed_kmh"].isna().sum())

print("Nearest gust station distance (km):")
print(fire_gust_merged["Nearest_Gust_Station_Dist_km"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# Should be 0 if all Climate_ID values exist in gust_stations
print((~fire_gust_merged["Climate_ID"].isin(gust_stations["Climate_ID"])).sum())

Rows: 2300610
Missing gust after merge (Max_Gust_Speed_kmh): 1289165
Nearest gust station distance (km):
count    2.300610e+06
mean     7.850037e+01
std      4.622711e+01
min      1.569962e-02
50%      6.839661e+01
90%      1.451978e+02
95%      1.605573e+02
99%      1.960564e+02
max      4.825750e+02
Name: Nearest_Gust_Station_Dist_km, dtype: float64
0


### Add distance flag columns

In [ ]:
# Based on median distances generate flag columns

# Core weather distance flags
fire_core_merged["Core_Dist_75km"] = (
    fire_core_merged["Nearest_Core_Station_Dist_km"] <= 75
)

fire_core_merged["Core_Dist_100km"] = (
    fire_core_merged["Nearest_Core_Station_Dist_km"] <= 100
)

# Gust weather distance flags
fire_gust_merged["Gust_Dist_75km"] = (
    fire_gust_merged["Nearest_Gust_Station_Dist_km"] <= 75
)

fire_gust_merged["Gust_Dist_100km"] = (
    fire_gust_merged["Nearest_Gust_Station_Dist_km"] <= 100
)

### Check and adjust columns before saving

In [ ]:
print("Fire-Core Merged Columns:", fire_core_merged.columns)
print("___________________________________________________________________________________")
print("Fire-Gust Merged Columns:", fire_gust_merged.columns)

Fire-Core Marged Columns: Index(['Fire_ID', 'Acquisition_Date', 'Acquisition_Time', 'Year', 'Month',
       'Day_of_Year', 'Hour', 'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4',
       'Bright_TI5', 'FRP', 'Scan', 'Track', 'Confidence', 'DayNight',
       'Fire_Type', 'LC_Type1_x', 'LC_Label_x', 'Confidence_Ordered',
       'detection_count', 'FRP_max', 'Date', 'Nearest_Core_Station_Dist_km',
       'Nearest_Core_Station_Index', 'Climate_ID', 'Station_Name', 'Province',
       'Elevation_m', 'Latitude_Weather', 'Longitude_Weather', 'Max_Temp_C',
       'Min_Temp_C', 'Mean_Temp_C', 'Total_Precip_mm', 'LC_Type1_y',
       'LC_Label_y', 'Core_Dist_75km', 'Core_Dist_100km'],
      dtype='object')
___________________________________________________________________________________
Fire-Gust Merged Columns: Index(['Fire_ID', 'Acquisition_Date', 'Acquisition_Time', 'Year', 'Month',
       'Day_of_Year', 'Hour', 'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4',
       'Bright_TI5', 'FRP', 'Sca

In [ ]:
# Clean up fire/core columns
fire_core_final = fire_core_merged.drop(
    columns=[
        "LC_Type1_y",
        "LC_Label_y",
        "Nearest_Core_Station_Index",
        "Acquisition_Date"
    ]
).rename(
    columns={
        "LC_Type1_x": "LC_Type1",
        "LC_Label_x": "LC_Label"
    }
)

print(fire_core_final.columns)
print(fire_core_final.shape)

# Clean up fire/gust columns
fire_gust_final = fire_gust_merged.drop(
    columns=[
        "LC_Type1_y",
        "LC_Label_y",
        "Nearest_Gust_Station_Index",
        "Acquisition_Date"
    ]
).rename(
    columns={
        "LC_Type1_x": "LC_Type1",
        "LC_Label_x": "LC_Label"
    }
)

print(fire_gust_final.columns)
print(fire_gust_final.shape)


Index(['Fire_ID', 'Acquisition_Time', 'Year', 'Month', 'Day_of_Year', 'Hour',
       'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4', 'Bright_TI5', 'FRP',
       'Scan', 'Track', 'Confidence', 'DayNight', 'Fire_Type', 'LC_Type1',
       'LC_Label', 'Confidence_Ordered', 'detection_count', 'FRP_max', 'Date',
       'Nearest_Core_Station_Dist_km', 'Climate_ID', 'Station_Name',
       'Province', 'Elevation_m', 'Latitude_Weather', 'Longitude_Weather',
       'Max_Temp_C', 'Min_Temp_C', 'Mean_Temp_C', 'Total_Precip_mm',
       'Core_Dist_75km', 'Core_Dist_100km'],
      dtype='object')
(2300610, 35)
Index(['Fire_ID', 'Acquisition_Time', 'Year', 'Month', 'Day_of_Year', 'Hour',
       'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4', 'Bright_TI5', 'FRP',
       'Scan', 'Track', 'Confidence', 'DayNight', 'Fire_Type', 'LC_Type1',
       'LC_Label', 'Confidence_Ordered', 'detection_count', 'FRP_max', 'Date',
       'Nearest_Gust_Station_Dist_km', 'Climate_ID', 'Station_Name',
       'Province', 

### Save final fire-weather merged datasets

In [ ]:
fire_core_final.to_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Fire_Core_Merged.csv", index=False)
fire_gust_final.to_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Fire_Gust_Merged.csv", index=False)

print("Core merged shape:", fire_core_final.shape)
print("Gust merged shape:", fire_gust_final.shape)

Core merged shape: (2300610, 35)
Gust merged shape: (2300610, 39)
